In [1]:
# === ColumnParallelLinear 教学版：按输出维度切分权重(省略 all-gather 通信) ===
import torch
import torch.nn as nn


class ColumnParallelLinear(nn.Module):
    """按输出维度切分的线性层(教学版)。

    标准 Linear 权重形状 [out_features, in_features],所有卡持有完整副本。
    ColumnParallel 把 out_features 切成 N 份,每张卡只持有 [out/N, in]。
    """

    def __init__(self, in_features, out_features, world_size):
        super().__init__()
        assert out_features % world_size == 0
        self.out_per_partition = out_features // world_size
        self.weight = nn.Parameter(
            torch.empty(self.out_per_partition, in_features)
        )
        nn.init.normal_(self.weight)

    def forward(self, x):
        # x: [batch, in_features] → 每张卡算 [batch, out/N]
        return x @ self.weight.t()


# 演示切分:8 维输出切到 4 张卡
world_size = 4
layer = ColumnParallelLinear(in_features=6, out_features=8, world_size=world_size)

print(f"标准 Linear 权重: [8, 6],共 {8*6} 个参数")
print(f"ColumnParallel 每卡权重: {list(layer.weight.shape)}")
print(f"  每卡持有 {layer.weight.numel()} 个参数 = 总量 / {world_size}")
print()
print("关键观察:TP 把权重切到 N 卡,每卡 forward 只算 [batch, out/N],")
print("        最后 all-gather 拼回 [batch, out](教学版省略了通信)。")

标准 Linear 权重: [8, 6],共 48 个参数
ColumnParallel 每卡权重: [2, 6]
  每卡持有 12 个参数 = 总量 / 4

关键观察:TP 把权重切到 N 卡,每卡 forward 只算 [batch, out/N],
        最后 all-gather 拼回 [batch, out](教学版省略了通信)。


In [1]:
# === RowParallelLinear 教学版：按输入维度切分权重，forward 产生部分和 ===
import torch
import torch.nn as nn


class RowParallelLinear(nn.Module):
    """按输入维度切分的线性层(教学版)。

    标准 Linear 权重形状 [out_features, in_features],所有卡持有完整副本。
    RowParallel 把 in_features 切成 N 份,每张卡持有 [out, in/N]。
    forward 时每卡产出 [batch, out] 的部分和,需要 all-reduce 求和。
    """

    def __init__(self, in_features, out_features, world_size):
        super().__init__()
        assert in_features % world_size == 0
        self.in_per_partition = in_features // world_size
        self.weight = nn.Parameter(
            torch.empty(out_features, self.in_per_partition)
        )
        nn.init.normal_(self.weight)

    def forward(self, x):
        # x: [batch, in/N] → [batch, out] 部分和
        return x @ self.weight.t()


# 配合 ColumnParallel + RowParallel 组合成完整 MLP（省略通信）
world_size = 4
in_dim, hidden_dim, out_dim = 6, 8, 6

# 上一节定义的 ColumnParallelLinear
col_layer = ColumnParallelLinear(in_dim, hidden_dim, world_size)
row_layer = RowParallelLinear(hidden_dim, out_dim, world_size)

print(f"ColumnParallel 权重: {list(col_layer.weight.shape)}  # [out/N, in]")
print(f"RowParallel    权重: {list(row_layer.weight.shape)}  # [out, in/N]")
print()

# 标准 Linear 的权重总量
standard_params = in_dim * hidden_dim + hidden_dim * out_dim
# TP 切分后的权重总量（每卡持有的权重之和 × 卡数）
col_per_card = col_layer.weight.numel()
row_per_card = row_layer.weight.numel()
tp_params = (col_per_card + row_per_card) * world_size

print(f"标准 MLP 权重总量: {standard_params}  (= {in_dim}×{hidden_dim} + {hidden_dim}×{out_dim})")
print(f"TP 每卡: col {col_per_card} + row {row_per_card} = {col_per_card + row_per_card}")
print(f"TP × {world_size} 卡 = {(col_per_card + row_per_card) * world_size}")
print(f"参数总量是否一致？ {standard_params == tp_params}")
print()
print("关键观察：TP 不改变参数总量，只是把矩阵切成 N 片。")
print("ColumnParallel [out, in] → N 片 [out/N, in]（按输出维度切）")
print("RowParallel    [out, in] → N 片 [out, in/N]（按输入维度切）")

ColumnParallel 权重: [2, 6]  # [out/N, in]
RowParallel    权重: [6, 2]  # [out, in/N]

标准 MLP 权重总量: 96  (= 6×8 + 8×6)
TP 每卡: col 12 + row 12 = 24
TP × 4 卡 = 96
参数总量是否一致？ True

关键观察：TP 不改变参数总量，只是把矩阵切成 N 片。
ColumnParallel [out, in] → N 片 [out/N, in]（按输出维度切）
RowParallel    [out, in] → N 片 [out, in/N]（按输入维度切）


上面两个教学版把 ColumnParallel 和 RowParallel 的切分逻辑拆开看了。Transformer 的 FFN 正好把两者串起来——第一层 ColumnParallel 升维、第二层 RowParallel 降维，中间的非线性在每张卡局部施加，整层只在结尾 all-reduce 一次。这和前面手算的结论一致：column → 激活 → row 路径只需一次通信。

工业界不会让用户每次手写这套切分——NVIDIA 的 Megatron-LM 把它封装成了生产级的 `ColumnParallelLinear` 和 `RowParallelLinear`。用户写的模型代码看起来和普通 PyTorch 几乎一样，底层自动按 TP 切分权重、插入通信。

**Megatron-LM 源码阅读路线**

Megatron-LM 不是 pip 安装的库，而是一套训练代码基线，代码量大。建议按这条路线读：

1. 入口 `pretrain_gpt.py`：训练循环骨架，理解一个 step 的 forward / backward / optimizer step 三段
2. 模型 `megatron/models/gpt/gpt_model.py`：GPTModel 如何组装 embedding + transformer blocks + lm_head
3. 并行层 `megatron/core/tensor_parallel/layers.py`：`ColumnParallelLinear` / `RowParallelLinear` 的实现，重点看权重如何按 rank 切分
4. 注意力 `megatron/core/transformer/attention.py`：TP 如何作用在 QKV projection(ColumnParallel)和 attention output(RowParallel)
5. 流水并行 `megatron/core/pipeline_parallel/schedules.py`：1F1B 调度如何交错 forward 和 backward
6. MoE `megatron/core/transformer/moe/`：专家并行如何与 TP、PP、DP 组合

读完前四步就能理解 TP 的全貌，第五步理解 PP 调度，第六步进入 MoE。这条路线也回答了「多维并行在工业代码里长什么样」——Megatron-LM 把 TP、PP、EP 都封进了自定义层和调度器，用户只在最外层配置并行度。

# 5D 并行策略全景

> 上一节把集合通信原语梳理了一遍：all-reduce 把梯度求和、all-gather 把分片参数拼起来、reduce-scatter 把梯度切片后聚合。有了这套语言，再回头看分布式训练的方案就不会被框架的各种封装绕晕——任何并行策略的本质都是「在哪些维度上切张量、在哪些时刻做哪些 collective」。
>
> 这一节把工业级训练用的五种并行维度全部摆开：Data Parallelism、Tensor Parallelism、Pipeline Parallelism、Sequence Parallelism、Expert Parallelism。每种都说清楚「切哪个轴、什么时候通信、为什么这样切省显存或省通信」，最后看它们如何组合成 Llama-3、DeepSeek-V3、Mixtral 的真实训练配置。


数据并行（Data Parallelism，DP）只切 batch，每张卡持完整模型副本，反向后 all-reduce 梯度；它把吞吐线性扩展到 N 倍，但显存完全没省。ZeRO 把 DP 里的冗余（优化器状态、梯度、参数）逐层切到多卡，单卡显存降到 16P/N，代价是引入额外的 all-gather 和 reduce-scatter。这部分在前面讲过，本附录作为串联只回顾结论。

真正扩展模型规模的钥匙是模型并行——把单层权重沿某个维度切开。Tensor Parallelism（TP）切 hidden 维度，前向只在一层内通信一次 all-reduce，反向再通信一次；Megatron-LM 把它推到了工业可用。Pipeline Parallelism（PP）切层，每张卡负责若干连续 transformer block，靠 micro-batch 流水掩盖通信；DeepSeek-V3 的 DualPipe 把流水做成双向进一步压低 bubble。Sequence Parallelism（SP）切序列长度，配合长上下文训练。Expert Parallelism（EP）专门用于 MoE，把不同 expert 分到不同卡，路由时用 all-to-all 交换 token。

本附录沿用上一节的 partition notation：用 $I_X$ 表示张量的 $I$ 轴沿设备网格的 $X$ 轴切分。代码用 numpy 在单进程里模拟多卡，每个 numpy 数组代表一张卡上的数据，重点是让读者看到「shape 怎么切」。


## 1. Data Parallelism 与 ZeRO：快速回顾

DDP 的四个步骤是：每张卡拿自己那份 micro-batch 做前向、反向算梯度、all-reduce 把梯度平均、每张卡独立更新完整参数。通信只在反向结尾发生一次 all-reduce。当模型放得下单卡时，DDP 是最优选择。

模型放不下时启用 ZeRO。三个 Stage 分别切优化器状态、梯度、参数。Stage 3 下每卡只持有 $1/N$ 的参数、梯度和优化器状态，前向时按需 all-gather 出当前层的完整权重，用完即丢。一个关键事实是：**三个 Stage 的通信量完全相等**。原因是 all-reduce 等价于 reduce-scatter + all-gather，ZeRO-1 把梯度 reduce-scatter 到多卡，Stage 3 把参数 all-gather 拼回，两次操作的通信量各占一半，加起来正好等于一次 all-reduce。所以从通信成本看，Stage 1/2/3 是免费的，只是用换回来的带宽换取了显存节省。

下面这个 cell 把三个 Stage 在 7B 模型、8 卡下的单卡显存再过一遍，作为本节的起点。


In [ ]:
# === ZeRO 三 Stage 单卡显存回顾：7B × 8 卡 ===
P = 7e9
N = 8

ddp_per_card   = 16 * P                       # 每卡完整副本
zero1_per_card = 2 * P + 2 * P + 12 * P / N   # 只切优化器
zero2_per_card = 2 * P + 2 * P / N + 12 * P / N
zero3_per_card = 16 * P / N                   # 全切

print(f"{'方案':<14}{'单卡 (GB)':>12}{'相对 DDP':>12}")
print("-" * 40)
for name, val in [("DDP",        ddp_per_card),
                  ("ZeRO-1",     zero1_per_card),
                  ("ZeRO-2",     zero2_per_card),
                  ("ZeRO-3",     zero3_per_card)]:
    print(f"{name:<14}{val/1e9:>10.1f}   {val/ddp_per_card*100:>8.1f}%")
print()
print("关键观察：三个 Stage 通信量相同，但单卡显存从 112 GB 压到 14 GB。")
print("代价是 Stage 越高，对带宽越敏感，对计算/通信比要求越高。")

## 2. Tensor Parallelism：切 hidden 维度

Tensor Parallelism（TP）的核心想法是把单层线性运算的权重沿某个维度切开，每张卡只算自己那一片。切法分两种：ColumnParallelLinear 把权重矩阵按列切（即按输出维度切），每张卡算出完整 batch 的部分列，无需通信；RowParallelLinear 把权重按行切（即按输入维度切），每张卡算出的是部分和，需要 all-reduce 才能得到完整结果。

Megatron-LM 的关键 insight 是把这两种切法串起来：一个 MLP 由两个线性层 $W_1$、$W_2$ 组成，$W_1$ 用 column parallel、$W_2$ 用 row parallel，中间的非线性激活在每张卡的局部列上独立施加。这样整层 MLP 从输入到输出只需要在结尾做一次 all-reduce。下面用手算展示这个性质为什么成立。

设 $h$ 是输入 hidden state（被复制到每张卡），$W_1$ 列切为 $[W_{1,0}; W_{1,1}]$，$W_2$ 行切为 $[W_{2,0}, W_{2,1}]^\top$。第一层后每张卡有 $h_i = h \cdot W_{1,i}$（局部，无需通信）；激活函数逐元素作用 $\sigma(h_i)$（仍局部）；第二层 $y = \sigma(h) \cdot W_2 = [\sigma(h_0), \sigma(h_1)] \cdot [W_{2,0}; W_{2,1}] = \sigma(h_0) \cdot W_{2,0} + \sigma(h_1) \cdot W_{2,1}$。两片加起来正好等于完整乘积，所以一次 all-reduce 就够。


In [ ]:
# === TP 切法手算：2 卡，验证 column -> row 只需一次 all-reduce ===
import numpy as np
np.random.seed(0)

d_model = 4     # 输入维度（小数字便于手算）
d_ff    = 6     # 中间维度
world_size = 2  # 2 张卡

# 构造完整权重（参考实现）
W1_full = np.random.randn(d_model, d_ff)      # (4, 6)
W2_full = np.random.randn(d_ff, d_model)      # (6, 4)
h       = np.random.randn(1, d_model)         # (1, 4)

def silu(x):
    return x / (1.0 + np.exp(-x))

# 参考输出：单卡上的完整计算（带 silu 激活，与 TP 路径一致）
y_ref = silu(h @ W1_full) @ W2_full           # (1, 4)

# TP 切法：W1 按列切（输出维度），W2 按行切（输入维度）
W1_shards = np.split(W1_full, world_size, axis=1)   # 2 片，每片 (4, 3)
W2_shards = np.split(W2_full, world_size, axis=0)   # 2 片，每片 (3, 4)

# 模拟每张卡的局部计算
partial = []
for rank in range(world_size):
    h_local = silu(h @ W1_shards[rank])       # (1, 3) - 无通信，激活局部施加
    y_partial = h_local @ W2_shards[rank]     # (1, 4) - 部分和
    partial.append(y_partial)
    print(f"rank {rank}: y_partial shape = {y_partial.shape}, 数值 = {y_partial.ravel()}")

# 模拟一次 all-reduce（求和）
y_tp = sum(partial)
print()
print(f"参考输出 y_ref = {y_ref.ravel()}")
print(f"TP 重建   y_tp = {y_tp.ravel()}")
print(f"最大误差 = {np.abs(y_ref - y_tp).max():.2e}")
print()
print("关键观察：column -> 激活 -> row 整条路径，只做了一次 all-reduce（求和）。")

### 2.1 Attention 按头切分

Transformer 的多头注意力里，每个 head 是独立的——head A 的 Q 只和 head A 的 K、V 做点积。这意味着如果按 head 把 QKV 切到不同卡上，每张卡可以独立完成自己那批 head 的 attention 计算，中间完全不需要通信。这是 TP 应用到 attention 时最优雅的地方：column parallel（按 head 切 QKV）→ 局部 attention → row parallel（输出投影），同样只在结尾一次 all-reduce。

工程上要保证 head 数能被 world_size 整除。比如 32 head、4 卡 TP，每卡处理 8 head，每卡 QKV 投影权重从 $d_\text{model} \times 3d_\text{model}$ 降为 $d_\text{model} \times 3 d_\text{model}/4$。


In [ ]:
# === Attention TP：按 head 切，验证免通信 ===
# 用「每个 head 独立计算后拼接、再投影」的标准实现做参考
d_model, num_heads = 8, 4
head_dim = d_model // num_heads      # 2
seq_len  = 3
world_size = 2                       # 2 卡，每卡 2 个 head

np.random.seed(2)
x = np.random.randn(1, seq_len, d_model)

# 每个 head 独立的 q/k/v/out 权重（这是标准 MHA 的等价写法）
W_qkv_per_head = np.random.randn(num_heads, d_model, 3 * head_dim)  # (4, 8, 6)
W_out_per_head = np.random.randn(num_heads, head_dim, d_model)      # (4, 2, 8)

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def head_forward(x, h_id):
    """单个 head 的完整前向。"""
    qkv = x @ W_qkv_per_head[h_id]               # (1, 3, 6) -> reshape
    qkv = qkv.reshape(1, seq_len, 3, head_dim)
    q, k, v = qkv[..., 0, :], qkv[..., 1, :], qkv[..., 2, :]
    # (1, seq, head_dim) -> (1, head_dim, seq) via transpose for q
    q = q.transpose(0, 2, 1)                     # (1, 2, 3)
    k = k.transpose(0, 2, 1)                     # (1, 2, 3)
    v = v.transpose(0, 2, 1)                     # (1, 2, 3)
    sc = q @ k.transpose(0, 2, 1) / np.sqrt(head_dim)
    at = softmax(sc) @ v                         # (1, 2, 3)
    at = at.transpose(0, 2, 1)                   # (1, 3, 2)
    return at @ W_out_per_head[h_id]             # (1, 3, 8) 部分和

# 参考：所有 head 求和（等价于完整 MHA）
y_ref = sum(head_forward(x, h) for h in range(num_heads))

# TP 切法：把 head 分到 2 张卡，每卡 2 个 head，零通信
heads_per_rank = num_heads // world_size
partials = []
for rank in range(world_size):
    local_heads = range(rank * heads_per_rank, (rank + 1) * heads_per_rank)
    partial = sum(head_forward(x, h) for h in local_heads)
    partials.append(partial)

y_tp = sum(partials)
print(f"参考输出 y_ref[0,0,:] = {y_ref[0,0,:]}")
print(f"TP 重建  y_tp [0,0,:] = {y_tp[0,0,:]}")
print(f"最大误差 = {np.abs(y_ref - y_tp).max():.2e}")
print()
print("关键观察：每张卡独立计算自己负责的 head，attention 内部零通信。")
print("结尾一次 all-reduce 把各卡部分和加起来。")

### 2.2 SwiGLU MLP 的三矩阵切分

Llama 系列用的 SwiGLU MLP 有三个权重矩阵：$W_\text{gate}$、$W_\text{up}$、$W_\text{down}$。前向是 $\text{SiLU}(x W_\text{gate}) \odot (x W_\text{up})$ 再乘 $W_\text{down}$。$W_\text{gate}$ 和 $W_\text{up}$ 都是 column parallel（按列切，每卡算自己那批中间维度的列），它们的输出逐元素相乘后仍沿中间维度分片；$W_\text{down}$ 用 row parallel，结尾一次 all-reduce。这和普通 MLP 的切法完全一致——只是中间多了个逐元素乘积。

MoE 模型里每个 expert 内部的 MLP 也是这套切法。也就是说 TP 既适用于 dense model 的 MLP，也适用于单个 expert 的 MLP——这点在第 5 节 Expert Parallelism 里会用到。


In [ ]:
# === SwiGLU MLP TP 切法：gate + up column, down row ===
d_model, d_ff = 4, 6
world_size = 2

W_gate = np.random.randn(d_model, d_ff)
W_up   = np.random.randn(d_model, d_ff)
W_down = np.random.randn(d_ff, d_model)
x      = np.random.randn(1, d_model)

def silu(z):
    return z / (1.0 + np.exp(-z))

# 参考：单卡完整计算
y_ref = (silu(x @ W_gate) * (x @ W_up)) @ W_down

# TP 切法
gate_sh = np.split(W_gate, world_size, axis=1)   # column
up_sh   = np.split(W_up,   world_size, axis=1)   # column
down_sh = np.split(W_down, world_size, axis=0)   # row

partials = []
for rank in range(world_size):
    g = x @ gate_sh[rank]                # (1, 3) 局部
    u = x @ up_sh[rank]                  # (1, 3) 局部
    h = silu(g) * u                       # 逐元素，仍局部
    y_partial = h @ down_sh[rank]        # (1, 4) 部分和
    partials.append(y_partial)

y_tp = sum(partials)
print(f"y_ref = {y_ref.ravel()}")
print(f"y_tp  = {y_tp.ravel()}")
print(f"最大误差 = {np.abs(y_ref - y_tp).max():.2e}")
print("SwiGLU 三矩阵：gate/up column + down row，结尾一次 all-reduce。")

### 2.3 TP 的通信代价

TP 的优点是单卡显存随 world_size 线性下降——权重切到 N 片，每卡只存 $1/N$。缺点是通信频繁且都在关键路径上：每个 transformer layer 的 MLP 和 Attention 各做一次 forward all-reduce 和一次 backward all-reduce，共 4 次。这些 all-reduce 跨节点时延敏感，所以 TP 通常只在一个节点内（8 张 NVLink 互联的卡）使用，跨节点交给 PP 或 DP。

下面这个表格对比 TP=1、TP=2、TP=4 时单层 MLP 的权重和通信次数（d_model=4096, d_ff=11008）：


In [ ]:
# === TP 规模 vs 单卡权重 / 通信次数 ===
d_model, d_ff = 4096, 11008
bytes_per_elem = 2   # FP16/BF16

print(f"{'TP':>4}{'单卡 MLP 权重 (MB)':>22}{'all-reduce 次数/层':>22}")
print("-" * 50)
for tp in [1, 2, 4, 8]:
    w_bytes = (d_model * d_ff + d_ff * d_model) * bytes_per_elem / tp
    comms = 4 if tp > 1 else 0   # fwd + bwd × (MLP + Attn)，TP=1 时无 collective
    print(f"{tp:>4}{w_bytes / 1e6:>20.1f}{comms:>22}")
print()
print("观察：TP=8 时单卡 MLP 权重压到 1/8，但每层 4 次 all-reduce。")
print("NVLink 内带宽足够，跨节点 TP 通常不划算，改用 PP 或 DP。")

### 2.4 TP 的代码封装：Megatron-LM

前面用 numpy 手算了 column→row 的切法，验证了整层 MLP 只需一次 all-reduce。工业界不会让用户每次手写这套切分——NVIDIA 的 Megatron-LM 把它封装成了两个标准的 `nn.Module` 子类：`ColumnParallelLinear` 和 `RowParallelLinear`。

`ColumnParallelLinear` 对应前面手算的 $W_1$：把权重按输出维度切到 N 张卡，每张卡只持有 `[out/N, in]` 那一片，forward 时各自算 `[batch, out/N]`，无需通信。下面是一个省略通信优化的教学版，把切分维度看清楚。

## 3. Pipeline Parallelism：切层

最朴素的模型并行是把每一层放到一张不同的卡上，前向时把 hidden state 从卡 0 送到卡 1 再送到卡 2……这种 naive 方式的问题在于：任何时刻只有一张卡在算，其它卡都在等。把 4 层分到 4 张卡上，吞吐和单卡相同，只是显存分摊了——这显然没有意义。

GPipe 的思路是把一个 mini-batch 切成 M 个 micro-batch，让它们在流水线里错峰执行：卡 0 算完 micro-batch 0 后立刻开始算 micro-batch 1，同时卡 1 在算 micro-batch 0。这样大多数时刻所有卡都在工作，只有流水线开头和结尾有空泡。空泡占比的公式是 $(P-1)/M$，其中 $P$ 是 stage 数（卡数），$M$ 是 micro-batch 数。$M$ 越大空泡越小。

1F1B（one forward, one backward）调度在 GPipe 基础上把反向也插进流水：每个 stage 算完一个 micro-batch 的前向后立刻算前一个 micro-batch 的反向，让激活值内存峰值下降，同时仍保持流水充满。


In [ ]:
# === GPipe bubble 占比手算 ===
print(f"{'P (stages)':>12}{'M (micro)':>12}{'bubble 比例':>14}")
print("-" * 40)
for P in [4, 8, 16]:
    for M in [8, 32, 128]:
        bubble = (P - 1) / M
        print(f"{P:>12}{M:>12}{bubble*100:>12.1f}%")
print()
print("观察：P=8、M=128 时空泡只占 5.5%，几乎可以忽略。")
print("实战中 M 通常选 stage 数的 4-8 倍以上来压低 bubble。")

### 3.1 1F1B 调度的流水线时间线

1F1B 的关键设计是让每个 stage 尽早开始反向。下面用字符画展示 P=4、M=4 的 1F1B 时间线：每个字母代表一个 micro-batch 的某次前向（数字）或反向（数字加撇）。stage 0 在最左边（最早），stage 3 在最右边。


In [ ]:
# === 1F1B 时间线字符画（P=4 stage, M=4 micro-batch） ===
P, M = 4, 4
print("1F1B 调度（F = 前向，B = 反向，数字是 micro-batch 编号）")
print("=" * 60)
# 简化的稳态段：每个 stage 一旦启动就交错做 F 和 B
for stage in range(P):
    # warmup：stage 越靠后，要等前面积累越多 micro-batch 才能开始
    warmup = stage
    slots = [" . "] * (warmup + M + (P - 1 - stage))
    # 前向段
    for mb in range(M):
        idx = warmup + mb
        if idx < len(slots):
            slots[idx] = f"F{mb}"
    # 反向段：紧贴最后一个前向之后
    for mb in range(M):
        idx = warmup + M + mb
        if idx < len(slots):
            slots[idx] = f"B{mb}"
    print(f"stage {stage}: " + " ".join(slots))
print()
print("关键观察：稳态时段每张卡都在干活，开头有 warmup 空泡、结尾有 cooldown 空泡。")
print("这正是 (P-1)/M 公式刻画的两端 bubble。")

### 3.2 DualPipe：双向流水（DeepSeek-V3）

1F1B 仍然有两端 bubble。DeepSeek-V3 提出的 DualPipe 把 micro-batch 分成两半，分别从流水线两端注入：一半从 stage 0 走到 stage P-1，另一半反向走。两条流水线在同一组卡上交错执行，每张卡同时承担一个方向的 forward 和另一个方向的 backward。这样 bubble 可以被对向的 micro-batch 计算掩盖，进一步压低空泡占比。

代价是每张卡要同时持有两个方向的激活值，显存峰值高于单向 1F1B。DeepSeek-V3 在 2048 张 H800 上训练时，PP=16，DualPipe 让 bubble 占比降到 1/P 量级，配合 EP+DP 完成了 14.8T 总 token 的预训练。这是目前工业界 PP 调度的代表性方案。


In [ ]:
# === DualPipe vs 1F1B bubble 对比（简化估算） ===
print(f"{'P':>4}{'M':>6}{'1F1B bubble':>16}{'DualPipe bubble':>20}")
print("-" * 48)
for P in [4, 8, 16]:
    M = 4 * P   # 实战中 M 常取 4P
    one_f1b = (P - 1) / M
    dual    = (P - 1) / (2 * M)   # 双向让有效 micro 数翻倍
    print(f"{P:>4}{M:>6}{one_f1b*100:>14.1f}%{dual*100:>18.1f}%")
print()
print("DualPipe 把有效 micro 数翻倍，bubble 减半。代价是激活值显存翻倍。")

## 4. Sequence Parallelism：切序列

Sequence Parallelism（SP）切的是序列长度维度。它有两个用途。第一是配合 TP：Megatron-LM 的 TP 里，LayerNorm 和 dropout 这类操作对每个 token 独立，没必要每张卡都持完整序列——把序列维度也切了，每卡只算自己那批 token 的 LayerNorm，在进入 TP 的 column/row parallel 之前用 all-gather 把序列拼回，在退出时用 reduce-scatter 重新切。这样 LayerNorm 和 dropout 的激活值显存按 N 切分，长序列训练时显著省显存。

第二个用途是真正的长上下文训练：当 seq_len 超过单卡显存上限时（比如 128K 或 1M context），即使模型本身能放下，attention 的中间激活也撑爆显存。这时把序列切到多卡，attention 计算时通过 ring attention 或 all-to-all 让每张卡看到自己那段序列与其它段的关系。这类方法包括 Ring Attention、Context Parallelism（DeepSpeed-Ulysses）等，本质都是 SP 在 attention 内部的精细化。


In [ ]:
# === SP：序列切分对 LayerNorm 激活显存的影响 ===
batch, seq_len, d_model = 1, 8192, 4096
bytes_per_elem = 2

full_act_bytes = batch * seq_len * d_model * bytes_per_elem

print(f"完整 LayerNorm 输入激活: {full_act_bytes / 1e6:.1f} MB")
print(f"{'SP':>6}{'单卡激活 (MB)':>18}{'节省比例':>12}")
print("-" * 40)
for sp in [1, 2, 4, 8]:
    per_card = full_act_bytes / sp
    print(f"{sp:>6}{per_card / 1e6:>16.1f}{(1 - 1/sp)*100:>10.0f}%")
print()
print("观察：SP=8 时单卡 LayerNorm 激活压到 1/8。")
print("代价：进入 TP 前要 all-gather 拼回，退出后要 reduce-scatter 切回去。")

## 5. Expert Parallelism：MoE 专用

Expert Parallelism（EP）专门用于 Mixture-of-Experts 模型。MoE 的核心结构是一组并行的 expert（每个 expert 是一个 MLP），router 为每个 token 选 top-k 个 expert 计算。EP 的做法是把 N 个 expert 分到 N 张卡上，每张卡只持有 1 个（或几个）expert 的权重。这与 DP 把同一个 expert 复制到所有卡刚好相反。

EP 的核心通信是 router 之后的 all-to-all：每个 token 选中的 expert 分布在不同卡上，所以前向时 token 要按路由结果发到对应 expert 所在的卡，算完后再 all-to-all 发回来。这是 EP 与 TP/DP 最大的区别——它用的是 all-to-all，不是 all-reduce。

DeepSeek-V3 用 256 个 expert、每 token 选 8 个，EP=64 把 expert 分到 64 张卡；Mixtral 8x7B 用 8 个 expert、EP=8。EP 还可以和 TP 组合：每张卡内部再做 TP，比如 EP=4 × TP=2 总共 8 张卡管 4 个 expert，每个 expert 内部权重切到 2 张卡。下面手算 EP 的 all-toall 通信量。


In [ ]:
# === EP mock：4 个 expert，4 张卡，每卡 1 个 expert ===
num_experts = 4
world_size  = 4
seq_len     = 8
d_model     = 4

np.random.seed(1)
# 每个 token 路由到某个 expert
tokens = np.random.randn(seq_len, d_model)
router_logits = tokens @ np.random.randn(d_model, num_experts)
assignments = router_logits.argmax(axis=1)   # 每个 token 选的 expert 编号

print(f"token -> expert 路由: {assignments.tolist()}")
print()

# 模拟 all-to-all dispatch：每张卡（expert）收集发向自己的 token
expert_inputs = [[] for _ in range(num_experts)]
for tok_idx, expert_id in enumerate(assignments):
    expert_inputs[expert_id].append(tokens[tok_idx])

for eid in range(num_experts):
    n_tokens = len(expert_inputs[eid])
    print(f"expert {eid} (卡 {eid}) 收到 {n_tokens} 个 token")
print()
print("关键观察：每个 token 只发到一张卡，all-toall 的通信量 = seq_len × d_model。")
print("计算完成后还要一次反向 all-to-all 把结果发回原 token 所在位置。")

In [ ]:
# === EP vs TP vs DP 在 MoE 上的切法对比 ===
# 假设 8 expert × 每个 expert MLP 权重 110M 参数，8 张卡
expert_params = 110e6
num_experts   = 8
total_params  = expert_params * num_experts
N             = 8

ep_per_card = expert_params     # EP: 每卡 1 个 expert
tp_per_card = total_params / N  # TP: 每个 expert 都切到 8 片，每卡每 expert 1/8
dp_per_card = total_params      # DP: 每卡完整副本

print(f"{'方案':<10}{'单卡 expert 权重 (M)':>22}{'路由后通信':>20}")
print("-" * 54)
print(f"{'EP':<10}{ep_per_card/1e6:>20.1f}{'all-to-all':>20}")
print(f"{'TP':<10}{tp_per_card/1e6:>20.1f}{'all-reduce':>20}")
print(f"{'DP':<10}{dp_per_card/1e6:>20.1f}{'all-reduce (grad)':>20}")
print()
print("EP 单卡最省，但 all-to-all 通信模式不同于 all-reduce，对网络拓扑要求高。")
print("大模型通常 EP + DP + TP 混用，下面第 6 节展开。")

## 6. 5D 并行组合

把五种并行维度拼起来就是工业级训练的「并行 recipe」。一个 2048 卡的集群通常组织成 3D 或 4D mesh：最内层 8 卡做 TP（NVLink 域内）、向外一组做 EP 或 PP（节点间）、再向外做 DP。下面是三个公开报告过的配置：

- **Llama-3 70B**：8 卡节点内 TP=8，节点间 PP=8，剩余维度做 DP，总 GPU 数 1024 左右。dense 模型无 EP。
- **Mixtral 8x7B**：8 expert + EP=8（每卡一个 expert），节点内 TP=1，DP 跨节点，约 128-256 卡训练。
- **DeepSeek-V3 671B（MoE，37B 激活）**：EP=64（256 expert 分到 64 卡，每卡 4 个）、TP=1（节点内不做 TP 因为 MLA 已让单层权重足够小）、PP=16 用 DualPipe、DP 把剩余 GPU 填满，总 GPU 2048 张 H800。

组合原则有三条：TP 只在节点内用（all-reduce 延迟敏感）；EP 配 MoE 且 expert 数和卡数匹配；PP 和 DP 填充剩余 GPU 并保证 micro-batch 数足够掩盖 bubble。模型越大、专家越多，EP 的占比越高。


In [ ]:
# === 典型并行 recipe 对比 ===
configs = [
    ("Llama-3 70B",   "dense",  {"TP": 8, "PP": 8,  "DP": 16, "EP": 1,  "SP": 1}),
    ("Mixtral 8x7B",  "MoE",    {"TP": 1, "PP": 1,  "DP": 16, "EP": 8,  "SP": 1}),
    ("DeepSeek-V3",   "MoE",    {"TP": 1, "PP": 16, "DP": 2,  "EP": 64, "SP": 1}),
]

print(f"{'模型':<18}{'类型':<8}{'TP':>4}{'PP':>5}{'DP':>5}{'EP':>5}{'SP':>5}{'总 GPU':>10}")
print("-" * 60)
for name, kind, c in configs:
    total = c["TP"] * c["PP"] * c["DP"] * c["EP"] * c["SP"]
    print(f"{name:<18}{kind:<8}{c['TP']:>4}{c['PP']:>5}{c['DP']:>5}{c['EP']:>5}{c['SP']:>5}{total:>10}")
print()
print("注意：DeepSeek-V3 的 EP=64 与 DP=2 是正交维度，"
      "EP 组内 64 张卡分 256 expert，DP 组间复制整组。")
print("总 GPU = TP × PP × DP × EP × SP，五个维度相乘。")

In [ ]:
# === 不同模型规模的典型 recipe ===
recipes = [
    ("7B dense",    {"TP": 1, "PP": 1, "DP": 8,  "EP": 1}),
    ("70B dense",  {"TP": 8, "PP": 4,  "DP": 4,  "EP": 1}),
    ("400B MoE",   {"TP": 4, "PP": 8,  "DP": 4,  "EP": 16}),
    ("1T+ MoE",    {"TP": 4, "PP": 16, "DP": 8,  "EP": 32}),
]

print(f"{'规模':<16}{'TP':>4}{'PP':>5}{'DP':>5}{'EP':>5}{'总 GPU':>10}")
print("-" * 50)
for name, c in recipes:
    total = c["TP"] * c["PP"] * c["DP"] * c["EP"]
    print(f"{name:<16}{c['TP']:>4}{c['PP']:>5}{c['DP']:>5}{c['EP']:>5}{total:>10}")
print()
print("规律：模型越大，PP 和 EP 占比越高，TP 通常锁定在节点内 4 或 8。")
print("DP 用于扩展 batch，但受 critical batch size 限制不能无限加。")

## 小结

确认你已经搞懂了这些：

- [ ] DDP 切 batch，all-reduce 在反向结尾；ZeRO 三个 Stage 通信量相等，只是用换回来的带宽换显存
- [ ] Tensor Parallelism 用 column parallel + row parallel 串联，整层 MLP/Attention 只需一次 forward all-reduce
- [ ] Attention 按 head 切分天然免通信，因为不同 head 之间独立
- [ ] Pipeline Parallelism 的 bubble 占比是 $(P-1)/M$，1F1B 和 DualPipe 进一步压低空泡
- [ ] Sequence Parallelism 切序列维度，配合 TP 省激活显存，独立用于长上下文训练
- [ ] Expert Parallelism 切 expert 维度，核心通信是 all-to-all，与 TP/DP 的 all-reduce 不同
- [ ] 5D 并行 = DP × TP × PP × SP × EP，总 GPU 数等于各维度乘积
- [ ] TP 锁节点内、PP/EP/DP 填充节点间，是工业级 recipe 的通用原则


## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

**作业 1：手算 7B 模型 TP=2 + DP=2 下的单卡显存**

7B dense 模型，AdamW + BF16 训练，4 张卡。配置 TP=2、DP=2，不开 ZeRO。每张卡持有完整优化器状态（DP 副本）但权重按 TP 切分。单卡固定显存是多少 GB？

小提示：TP=2 把每层权重切到 2 张卡，所以参数部分每卡是 $2P/2$；梯度同理。优化器状态部分没有 ZeRO，所以仍是 $12P$。总计 $2P/2 + 2P/2 + 12P = 14P$ bytes。


In [ ]:
# 作业 1：TP=2 + DP=2 下 7B 模型的单卡显存
P = 7e9

# TODO: 计算单卡显存（bytes）
# TP=2 切权重和梯度，DP=2 不切优化器状态
tp_per_card_bytes = None

assert tp_per_card_bytes is not None, "请先计算单卡显存"
expected = 2 * P / 2 + 2 * P / 2 + 12 * P   # = 14P
assert abs(tp_per_card_bytes - expected) < 1e9, f"应为 {expected / 1e9:.0f} GB"
print(f"作业 1 通过：")
print(f"   TP=2 + DP=2 下 7B 模型单卡显存 = {tp_per_card_bytes / 1e9:.0f} GB")
print(f"   对比纯 DDP 的 112 GB，省了 {(112 - tp_per_card_bytes/1e9):.0f} GB（权重切了一半）。")

**作业 2：算 PP=8、M=32 时的 bubble 占比**

小提示：bubble = (P-1)/M。P=8、M=32 代入即可。


In [ ]:
# 作业 2：PP bubble 占比
P, M = 8, 32

# TODO: 计算 bubble 占比（0-1 之间的小数）
bubble_ratio = None

assert bubble_ratio is not None, "请先计算 bubble 占比"
expected = (P - 1) / M
assert abs(bubble_ratio - expected) < 1e-6, f"应为 {expected:.4f}"
print(f"作业 2 通过：")
print(f"   PP={P}、M={M} 时 bubble 占比 = {bubble_ratio*100:.1f}%")
print(f"   如果改成 DualPipe（双向），有效 M 翻倍，bubble 降到 {bubble_ratio*50:.1f}%。")

**作业 3：补全 DeepSeek-V3 风格的 5D 并行总 GPU 数**

DeepSeek-V3 配置 TP=1、PP=16、DP=2、EP=64、SP=1。算出总 GPU 数。

小提示：5D 并行总 GPU = TP × PP × DP × EP × SP。


In [ ]:
# 作业 3：DeepSeek-V3 总 GPU 数
config = {"TP": 1, "PP": 16, "DP": 2, "EP": 64, "SP": 1}

# TODO: 计算总 GPU 数
total_gpus = None

assert total_gpus is not None, "请先计算总 GPU 数"
expected = config["TP"] * config["PP"] * config["DP"] * config["EP"] * config["SP"]
assert total_gpus == expected, f"应为 {expected}"
print(f"作业 3 通过：")
print(f"   DeepSeek-V3 配置 {config}")
print(f"   总 GPU = {total_gpus}（实际公开报告为 2048 张 H800）")
print(f"   EP=64 占用了大部分卡，因为 256 个 expert 分到 64 张卡每卡 4 个 expert。")

## 参考资料

- Shoeybi et al., [Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism](https://arxiv.org/abs/1909.08053), 2019
- Huang et al., [GPipe: Efficient Training of Giant Neural Networks using Pipeline Parallelism](https://arxiv.org/abs/1811.06965), 2018
- Narayanan et al., [Memory-Efficient Pipeline-Parallel DNN Training (1F1B)](https://arxiv.org/abs/2004.13378), 2020
- DeepSeek-AI, [DeepSeek-V3 Technical Report (DualPipe)](https://arxiv.org/abs/2412.19437), 2024
- Korthikanti et al., [Reducing Activation Recomputation in Large Transformer Models (Sequence Parallelism)](https://arxiv.org/abs/2205.05198), 2022
- Fedus et al., [Switch Transformers (Expert Parallelism)](https://arxiv.org/abs/2101.03961), 2021
- Liu et al., [Ring Attention with Blockwise Transformers for Near-Infinite Context](https://arxiv.org/abs/2310.01889), 2023
